In [ ]:
import os, sys, json, time, random, warnings


_SRC_DIR = '/kaggle/input/datasets/muhammadaneeq786/src-new2/src'
assert os.path.isdir(_SRC_DIR), (
    f'MISSING: src-new2/src not found at {_SRC_DIR}\n'
    'Add dataset muhammadaneeq786/src-new2 to Kaggle inputs.'
)
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)
print(f'src-new2: FOUND at {_SRC_DIR}')
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from collections import Counter
from concurrent.futures import ThreadPoolExecutor


!pip install -q radgraph 2>/dev/null

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")


SEED = 9223
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
MODULE2_DIR  = "/kaggle/input/datasets/muhammadaneeq786/module2-outputs"
ANNOTATION   = "/kaggle/input/datasets/muhammadaneeq786/annotation/annotation.json"
PREDICTIONS  = os.path.join(MODULE2_DIR, "predictions_baseline.csv")
TEST_SAMPLES = os.path.join(MODULE2_DIR, "test_samples.csv")
OUTPUT_DIR   = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


for _label, _path in [
    ("module2-outputs/predictions_baseline.csv", PREDICTIONS),
    ("module2-outputs/test_samples.csv",          TEST_SAMPLES),
    ("annotation/annotation.json",                ANNOTATION),
]:
    assert os.path.exists(_path), (
        f"MISSING REQUIRED FILE: {_label}\n"
        f"  Expected: {_path}\n"
        "  Add the corresponding dataset to Kaggle inputs and re-run."
    )
    print(f"  FOUND {_label}")


predictions_df  = pd.read_csv(PREDICTIONS)
test_samples_df = pd.read_csv(TEST_SAMPLES)
print(f"\nLoaded {len(predictions_df)} generated reports")
print(f"  Columns: {list(predictions_df.columns)}")




print("\nLoading ground truth from annotation.json ...")
with open(ANNOTATION) as _f:
    _annot = json.load(_f)

_gt_lookup: dict = {}
for _split in ('train', 'val', 'test'):
    for _entry in _annot.get(_split, []):
        _sid  = str(_entry.get('study_id', '')).strip()
        _text = str(_entry.get('report', '')).strip()
        if _sid and _text:
            _gt_lookup[_sid]       = _text   
            _gt_lookup['s' + _sid] = _text   

assert _gt_lookup, (
    "annotation.json loaded but contains no valid study_id/report entries.\n"
    "Expected: {train/val/test: [{study_id: int, report: str, ...}]}"
)
print(f"  Indexed {len(_gt_lookup)//2:,} study reports ({len(_gt_lookup):,} keys incl. s-prefix)")

if 'ground_truth' not in predictions_df.columns:
    predictions_df['ground_truth'] = ''

def _fill_gt(row):
    cur = str(row.get('ground_truth', '')).strip()
    if cur and cur.lower() != 'nan':
        return cur
    return _gt_lookup.get(str(row.get('study_id', '')).strip(), '')

predictions_df['ground_truth'] = predictions_df.apply(_fill_gt, axis=1)
_n_gt = (predictions_df['ground_truth'].astype(str).str.strip().replace('nan', '') != '').sum()

assert _n_gt > 0, (
    f"STOP: Ground truth populated for 0/{len(predictions_df)} reports.\n"
    f"  Sample prediction study_ids: {predictions_df['study_id'].iloc[:5].tolist()}\n"
    f"  Sample annotation keys:      {list(_gt_lookup.keys())[:6]}\n"
    "  study_id format mismatch — fix and re-run."
)
print(f"  Ground truth: {_n_gt}/{len(predictions_df)} reports populated")


metrics_path = os.path.join(MODULE2_DIR, "baseline_metrics.json")
assert os.path.exists(metrics_path), (
    f"MISSING: {metrics_path}\n"
    "baseline_metrics.json must be present in module2-outputs."
)
with open(metrics_path) as f:
    m2_metrics = json.load(f)
print("\n--- Module 2 Baseline Scores ---")
for k, v in m2_metrics.items():
    try:
        print(f"  {k}: {float(v):.4f}")
    except (ValueError, TypeError):
        print(f"  {k}: {v}")

print("\n--- Sample Report ---")
_s = predictions_df.iloc[0]
print(f"  ID:  {_s['study_id']}")
print(f"  GEN: {str(_s['report'])[:120]}...")
print(f"  GT:  {str(_s['ground_truth'])[:120]}...")

In [ ]:
!pip install -q "transformers<4.40.0"

In [ ]:
from radgraph import RadGraph
import torch
import time




NUM_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {NUM_GPUS}")

start = time.time()
rg_models = []

try:
    if NUM_GPUS >= 2:
        print("Loading RadGraph-XL on GPU 0...")
        rg_gpu0 = RadGraph(model_type="radgraph-xl", cuda=0)
        print("Loading RadGraph-XL on GPU 1...")
        rg_gpu1 = RadGraph(model_type="radgraph-xl", cuda=1)
        rg_models = [rg_gpu0, rg_gpu1]
        print(f"\n✅ Dual-GPU RadGraph loaded in {time.time()-start:.1f}s")
    elif NUM_GPUS == 1:
        print("Loading RadGraph-XL on GPU 0...")
        rg_gpu0 = RadGraph(model_type="radgraph-xl", cuda=0)
        rg_models = [rg_gpu0]
        print(f"\n✅ Single-GPU RadGraph loaded in {time.time()-start:.1f}s")
    else:
        print("Loading RadGraph-XL on CPU...")
        rg_gpu0 = RadGraph(model_type="radgraph-xl", cuda=-1)
        rg_models = [rg_gpu0]
        print(f"\n✅ CPU RadGraph loaded in {time.time()-start:.1f}s")
except Exception as e:
    print(f"\n❌ Failed to load RadGraph: {e}")
    print("TIP: Try running '!pip install \"transformers<4.40.0\"' and restart the kernel.")

In [ ]:
reports   = predictions_df['report'].tolist()
study_ids = predictions_df['study_id'].tolist()


reports = [r.strip() if isinstance(r, str) and r.strip() else "normal" for r in reports]

print(f"Processing {len(reports)} reports on {len(rg_models)} GPU(s)...")
start = time.time()

def extract_chunk(model, chunk):
    
    return model(chunk)

if len(rg_models) >= 2:
    
    mid = len(reports) // 2
    chunks = [reports[:mid], reports[mid:]]

    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = [
            executor.submit(extract_chunk, rg_models[i], chunks[i])
            for i in range(2)
        ]
        results = [f.result() for f in futures]

    
    all_annotations = {}
    for k, v in results[0].items():
        all_annotations[str(int(k))] = v
    offset = len(chunks[0])
    for k, v in results[1].items():
        all_annotations[str(int(k) + offset)] = v
else:
    
    all_annotations = rg_models[0](reports)

elapsed = time.time() - start
print(f"\n✅ Extracted entities from {len(all_annotations)} reports in {elapsed:.1f}s")
print(f"   ({elapsed/len(reports)*1000:.0f} ms/report)")


total_entities = sum(len(v.get('entities', {})) for v in all_annotations.values())
print(f"   Total entities: {total_entities}")
print(f"   Avg entities/report: {total_entities/len(all_annotations):.1f}")


sample = all_annotations["0"]
print(f"\n--- Sample: Report 0 ---")
print(f"Text: {sample['text'][:120]}...")
print(f"Entities ({len(sample['entities'])}):")
for eid, ent in list(sample['entities'].items())[:6]:
    rels = [f"{r[0]}→e{r[1]}" for r in ent['relations']] if ent['relations'] else []
    print(f"  [{eid}] '{ent['tokens']}' — {ent['label']}"
          f"{' | ' + ', '.join(rels) if rels else ''}")

In [ ]:
REGION_NAMES = {
    0: 'Right Lung',   1: 'Left Lung',    2: 'Heart/Cardiac',
    3: 'Mediastinum',  4: 'Bone/Skeleton', 5: 'Pleural',
    6: 'Diaphragm',    7: 'Global/Other',
}

import glob as _glob
from collections import defaultdict




IMAGENOME_OBJ_TO_REGION = {
    
    'right lung': 0,           'right upper lobe': 0,
    'right mid lung zone': 0,  'right lower lung zone': 0,
    'right apical zone': 0,
    
    'left lung': 1,            'left upper lobe': 1,
    'left mid lung zone': 1,   'left lower lung zone': 1,
    'left apical zone': 1,
    
    'cardiac silhouette': 2,   'right atrium': 2,   'left ventricle': 2,
    
    'upper mediastinum': 3,    'mediastinum': 3,    'trachea': 3,
    'aortic arch': 3,          'right hilar structures': 3,
    'left hilar structures': 3,
    
    'spine': 4,       'right shoulder': 4,  'left shoulder': 4,
    'right clavicle': 4,       'left clavicle': 4,
    
    'right costophrenic angle': 5,  'left costophrenic angle': 5,
    
    'right hemidiaphragm': 6,  'left hemidiaphragm': 6,
    
    'abdomen': 7,  'neck': 7,
}


CHEST_IMAGENOME_DIR = "/kaggle/input/datasets/muhammadaneeq786/chest-imagenome-scene-graph"
assert os.path.isdir(CHEST_IMAGENOME_DIR), (
    f"MISSING: Chest ImaGenome not found at {CHEST_IMAGENOME_DIR}\n"
    "Add dataset muhammadaneeq786/chest-imagenome-scene-graph to Kaggle inputs."
)
imagenome_available = True
print(f"Chest ImaGenome: FOUND at {CHEST_IMAGENOME_DIR}")

OBS_REGION_TABLE_EMPIRICAL: dict = {}   
imagenome_study_regions:    dict = {}   
imagenome_obs_region_freq:  dict = {}   
imagenome_n_loaded:         int  = 0
imagenome_n_sg_files:       int  = 0

if imagenome_available:
    
    sg_files: list = []
    
    for sg_pattern in [
        
        os.path.join(CHEST_IMAGENOME_DIR, "scene_graph", "*.json"),
        
        os.path.join(CHEST_IMAGENOME_DIR, "**", "scene_graph", "**", "*.json"),
        os.path.join(CHEST_IMAGENOME_DIR, "**", "*.json"),
    ]:
        sg_files = _glob.glob(sg_pattern, recursive=True)
        
        sg_files = [f for f in sg_files
                    if not os.path.basename(f).startswith('.')
                    and os.path.getsize(f) > 100]
        if len(sg_files) > 100:   
            break

    imagenome_n_sg_files = len(sg_files)
    print(f"  Scene graph JSON files found: {imagenome_n_sg_files:,}")

    assert imagenome_n_sg_files > 0, (
        f"STOP: 0 scene graph JSON files found under {CHEST_IMAGENOME_DIR}\n"
        "Expected: {dataset}/scene_graph/*.json\n"
        "Check the dataset mount and re-run."
    )
    if imagenome_n_sg_files > 0:  
        
        
        
        _test_study_set: set = set()
        for _sid in study_ids:
            _s = str(_sid).strip()
            _test_study_set.add(_s)
            _numeric = _s.lstrip('sS')   
            if _numeric != _s:
                _test_study_set.add(_numeric)

        
        
        _obs_freq: dict             = defaultdict(Counter)
        _study_regions_raw: dict    = {}
        _n_loaded: int              = 0
        _n_errors: int              = 0

        for fpath in tqdm(sg_files, desc="Loading ImaGenome scene graphs",
                          unit="sg", mininterval=10.0):
            try:
                with open(fpath, 'r') as _f:
                    sg = json.load(_f)
            except Exception:
                _n_errors += 1
                continue

            study_id_sg = str(sg.get('study_id', '')).strip()
            
            study_id_sg_s = 's' + study_id_sg if not study_id_sg.startswith('s') else study_id_sg
            study_id_sg_plain = study_id_sg.lstrip('sS')  
            objects     = sg.get('objects', [])
            attributes  = sg.get('attributes', [])
            
            if isinstance(attributes, dict):
                attributes = list(attributes.values())

            
            
            
            _obj_to_region: dict = {}
            for obj in objects:
                bbox      = str(obj.get('bbox_name', '')).lower().strip()
                region_id = IMAGENOME_OBJ_TO_REGION.get(bbox, -1)
                if region_id < 0:   
                    name = str(obj.get('name', '')).lower().strip()
                    region_id = IMAGENOME_OBJ_TO_REGION.get(name, -1)
                if region_id >= 0:
                    _obj_to_region[str(obj.get('object_id', ''))] = region_id

            
            
            
            for attr_obj in (attributes if isinstance(attributes, list) else []):
                if not isinstance(attr_obj, dict):
                    continue
                obj_id    = str(attr_obj.get('object_id', ''))
                region_id = _obj_to_region.get(obj_id, -1)
                if region_id < 0:
                    
                    bbox = str(attr_obj.get('bbox_name', '')).lower().strip()
                    region_id = IMAGENOME_OBJ_TO_REGION.get(bbox, -1)
                if region_id < 0:
                    continue
                attr_pairs = attr_obj.get('attributes', [])
                if not isinstance(attr_pairs, list):
                    continue

                for attr in attr_pairs:
                    if not (isinstance(attr, (list, tuple)) and len(attr) >= 2):
                        continue
                    finding  = str(attr[0]).lower().strip()
                    presence = str(attr[1]).lower().strip()
                    is_present = presence in ('yes', 'present', 'true', '1', 'positive', 'y')

                    
                    if not finding or finding in (
                        'normal', 'clear', 'unremarkable', 'na', 'n/a', ''
                    ):
                        continue

                    
                    if is_present:
                        _obs_freq[finding][region_id] += 1

                    
                    if is_present and (
                study_id_sg in _test_study_set or
                study_id_sg_s in _test_study_set or
                study_id_sg_plain in _test_study_set
            ):
                        
                        _sid_key = study_id_sg_s if study_id_sg_s in _test_study_set else study_id_sg
                        if _sid_key not in _study_regions_raw:
                            _study_regions_raw[_sid_key] = {}
                        if region_id not in _study_regions_raw[_sid_key]:
                            _study_regions_raw[_sid_key][region_id] = set()
                        _study_regions_raw[_sid_key][region_id].add(finding)

            _n_loaded += 1

        imagenome_n_loaded        = _n_loaded
        imagenome_obs_region_freq = dict(_obs_freq)
        
        imagenome_study_regions = {
            sid: {r: sorted(f) for r, f in rd.items()}
            for sid, rd in _study_regions_raw.items()
        }

        
        
        
        
        
        
        
        for obs_term, region_counter in _obs_freq.items():
            total = sum(region_counter.values())
            if total < 50:
                continue   
            top_regions = sorted(
                [r for r, c in region_counter.items() if c / total >= 0.05],
                key=lambda r: -region_counter[r]   
            )
            if top_regions:
                OBS_REGION_TABLE_EMPIRICAL[obs_term] = top_regions

        
        print(f"\n{'='*65}")
        print("CHEST IMAGENOME — LOADING SUMMARY")
        print(f"{'='*65}")
        print(f"  Scene graphs loaded:             {imagenome_n_loaded:,} / {imagenome_n_sg_files:,}")
        print(f"  Parse errors (skipped):          {_n_errors:,}")
        print(f"  Test-set studies in ImaGenome:   "
              f"{len(imagenome_study_regions)} / {len(_test_study_set)}")
        print(f"  Unique observation terms indexed: {len(_obs_freq):,}")
        print(f"  Empirical obs→region entries:    {len(OBS_REGION_TABLE_EMPIRICAL)}")
        print(f"  (min-support ≥50 occurrences; region threshold ≥5%)")

        if OBS_REGION_TABLE_EMPIRICAL:
            print(f"\n  Top-10 empirical observation-to-region mappings:")
            print(f"  {'Observation':<30}  {'Regions':<45}  Occurrences")
            print(f"  {'-'*80}")
            _sorted_emp = sorted(
                OBS_REGION_TABLE_EMPIRICAL.items(),
                key=lambda x: -sum(_obs_freq[x[0]].values())
            )[:10]
            for term, rids in _sorted_emp:
                rnames = ', '.join(REGION_NAMES.get(r, f'R{r}') for r in rids)
                n_total = sum(_obs_freq.get(term, Counter()).values())
                region_pcts = '  '.join(
                    f"R{r}:{_obs_freq[term][r]/n_total*100:.0f}%"
                    for r in rids
                )
                print(f"  {term:<30}  {rnames:<45}  n={n_total:,}  ({region_pcts})")

        if imagenome_study_regions:
            _sid_ex = next(iter(imagenome_study_regions))
            print(f"\n  Example test study ({_sid_ex}) ImaGenome GT regions:")
            for rid, findings in list(imagenome_study_regions[_sid_ex].items())[:4]:
                rname = REGION_NAMES.get(int(rid), f'R{rid}')
                print(f"    Region {rid} ({rname}): {findings[:5]}")

In [ ]:
REGION_NAMES = {
    0: "Right Lung",
    1: "Left Lung",
    2: "Heart/Cardiac",
    3: "Mediastinum",
    4: "Bone/Skeleton",
    5: "Pleural",
    6: "Diaphragm",
    7: "Global/Other",
}


ANATOMY_REGION_TABLE = {
    
    'right lung': 0, 'right lower lobe': 0, 'right upper lobe': 0,
    'right middle lobe': 0, 'right hilum': 0, 'right apex': 0,
    'right base': 0, 'right hemithorax': 0,
    
    'left lung': 1, 'left lower lobe': 1, 'left upper lobe': 1,
    'left hilum': 1, 'left apex': 1, 'left base': 1,
    'left hemithorax': 1, 'lingula': 1,
    
    'heart': 2, 'cardiac': 2, 'cardiac silhouette': 2,
    'cardiomediastinal silhouette': 2, 'pericardial': 2,
    'pericardium': 2, 'cardiomegaly': 2, 'atrium': 2, 'ventricle': 2,
    
    'mediastinum': 3, 'mediastinal': 3, 'cardiomediastinal': 3,
    'aorta': 3, 'aortic': 3, 'hilar': 3, 'hila': 3,
    'vascular': 3, 'vasculature': 3, 'pulmonary vasculature': 3,
    'vascular pedicle': 3, 'trachea': 3, 'carina': 3,
    
    'spine': 4, 'thoracic spine': 4, 'osseous': 4, 'bone': 4,
    'bony': 4, 'rib': 4, 'ribs': 4, 'skeletal': 4, 'vertebral': 4,
    'vertebra': 4, 'clavicle': 4, 'shoulder': 4, 'sternum': 4,
    'scapula': 4, 'fracture': 4, 'scoliosis': 4,
    
    'pleural': 5, 'costophrenic': 5, 'costophrenic angle': 5,
    'pleura': 5, 'pleural space': 5,
    'pneumothorax': 5, 'effusion': 5, 'pleural effusion': 5,
    
    'diaphragm': 6, 'hemidiaphragm': 6, 'diaphragmatic': 6,
    'right hemidiaphragm': 6, 'left hemidiaphragm': 6,
    
    'tube': 7, 'catheter': 7, 'pacemaker': 7, 'port': 7,
    'line': 7, 'drain': 7, 'airway': 7, 'bronchus': 7,
    'bronchi': 7, 'bronchial': 7,
}


OBSERVATION_REGION_TABLE = {
    'opacity': [0, 1], 'opacities': [0, 1], 'consolidation': [0, 1],
    'infiltrate': [0, 1], 'atelectasis': [0, 1], 'pneumonia': [0, 1],
    'edema': [0, 1], 'pulmonary edema': [0, 1], 'fibrosis': [0, 1],
    'emphysema': [0, 1], 'nodule': [0, 1], 'mass': [0, 1],
    'granuloma': [0, 1], 'calcification': [0, 1], 'thickening': [0, 1],
    'density': [0, 1], 'lesion': [0, 1], 'congestion': [0, 1],
    'cardiomegaly': [2], 'effusion': [5], 'pleural effusion': [5],
    'pneumothorax': [5], 'fracture': [4], 'scoliosis': [4],
}


BILATERAL_LUNG_TERMS = {
    'lung', 'lungs', 'pulmonary', 'parenchyma', 'bilateral',
    'bibasilar', 'basilar', 'lower lobes', 'upper lobes',
    'lobes', 'apices', 'bases',
}


def map_anatomy_to_region(token_text):
    
    text = token_text.lower().strip()

    
    if text in ANATOMY_REGION_TABLE:
        return [ANATOMY_REGION_TABLE[text]]

    
    if text in BILATERAL_LUNG_TERMS:
        return [0, 1]

    
    
    if text in OBS_REGION_TABLE_EMPIRICAL:
        return list(OBS_REGION_TABLE_EMPIRICAL[text])

    
    if text in OBSERVATION_REGION_TABLE:
        return list(OBSERVATION_REGION_TABLE[text])

    
    for key, region in ANATOMY_REGION_TABLE.items():
        if key in text or text in key:
            return [region]

    
    for key, region_ids in OBSERVATION_REGION_TABLE.items():
        if key in text or text in key:
            return list(region_ids)

    
    if 'right' in text:
        return [0]
    if 'left' in text:
        return [1]
    if any(w in text for w in ('chest', 'thorax', 'thoracic')):
        return [0, 1]

    return [-1]  



_first_key = next(iter(all_annotations))
_first_ann  = all_annotations[_first_key]
_sample_labels = [ent['label'] for ent in list(_first_ann.get('entities', {}).values())[:5]]
print(f"DEBUG — RadGraph label format (first report): {_sample_labels}")



structured_output = []
imagenome_sourced_count = 0   

for idx in range(len(reports)):
    key = str(idx)
    ann = all_annotations.get(key, {'entities': {}, 'text': reports[idx]})
    study_id = study_ids[idx]
    report_text = reports[idx]

    
    anatomy_map = {}  
    entities_parsed = []

    for eid, ent in ann.get('entities', {}).items():
        label  = ent['label']
        tokens = ent['tokens']
        
        
        
        
        
        
        if '::' in label:
            _type_str, tag = label.split('::', 1)
            tag      = tag.lower().strip() or 'unknown'
            raw_type = _type_str.upper().strip()
        else:
            label_norm = label.upper().replace('_', '-')
            parts    = label_norm.split('-', 1)
            raw_type = parts[0].strip() if parts else 'UNK'
            raw_tag  = parts[1].strip() if len(parts) > 1 else ''
            _tag_map = {'DP': 'definitely present', 'DA': 'definitely absent', 'U': 'uncertain'}
            tag = _tag_map.get(raw_tag, raw_tag.lower() or 'unknown')
        if 'ANAT' in raw_type:
            entity_type = 'ANAT'
        elif 'OBS' in raw_type:
            entity_type = 'OBS'
        else:
            entity_type = raw_type

        regions = []
        if entity_type == 'ANAT':
            regions = map_anatomy_to_region(tokens)
            
            anatomy_map[str(eid)] = {'tokens': tokens, 'regions': regions}

        entities_parsed.append({
            'entity_id': eid,
            'tokens': tokens,
            'type': entity_type,        
            'tag': tag,                  
            'start_ix': ent.get('start_ix', -1),
            'end_ix': ent.get('end_ix', -1),
            'regions': regions,
            'relations': ent.get('relations', []),
        })

    
    for ep in entities_parsed:
        if ep['type'] == 'OBS' and (not ep['regions'] or ep['regions'] == [-1]):
            for rel in ep['relations']:
                if rel[0] == 'located_at':
                    target = str(rel[1])  
                    if target in anatomy_map:
                        ep['regions'] = anatomy_map[target]['regions']
                        break

    
    
    
    
    
    for ep in entities_parsed:
        if not ep['regions'] or ep['regions'] == [-1]:
            fallback = map_anatomy_to_region(ep['tokens'])
            if fallback != [-1]:
                ep['regions'] = fallback

    
    
    
    if imagenome_study_regions:
        sid_str = str(study_id).strip()
        
        sid_key = (sid_str if sid_str in imagenome_study_regions
                   else ('s' + sid_str if 's' + sid_str in imagenome_study_regions
                         else None))
        if sid_key is not None:
            _sg_data = imagenome_study_regions[sid_key]  
            _f2r: dict = defaultdict(list)
            for _rid, _flist in _sg_data.items():
                _rid_int = int(_rid) if isinstance(_rid, str) else _rid
                for _f in _flist:
                    _f2r[_f].append(_rid_int)
            for ep in entities_parsed:
                if ep['type'] != 'OBS':
                    continue
                if ep['regions'] and ep['regions'] != [-1]:
                    continue   
                finding = ep['tokens'].lower().strip()
                _matched = []
                for sg_f, sg_rids in _f2r.items():
                    if finding == sg_f or finding in sg_f or sg_f in finding:
                        _matched.extend(sg_rids)
                if _matched:
                    ep['regions'] = list(set(_matched))
                    ep['imagenome_sourced'] = True
                    imagenome_sourced_count += 1

    structured_output.append({
        'study_id': study_id,
        'report': report_text,
        'num_entities': len(entities_parsed),
        'entities': entities_parsed,
    })


total_ent     = sum(s['num_entities'] for s in structured_output)
anatomy_count = sum(1 for s in structured_output for e in s['entities']
                    if e['type'] == 'ANAT')
obs_present   = sum(1 for s in structured_output for e in s['entities']
                    if 'present' in e['tag'] and e['type'] == 'OBS')
obs_absent    = sum(1 for s in structured_output for e in s['entities']
                    if 'absent' in e['tag'] and e['type'] == 'OBS')
obs_uncertain = sum(1 for s in structured_output for e in s['entities']
                    if 'uncertain' in e['tag'] and e['type'] == 'OBS')
with_region   = sum(1 for s in structured_output for e in s['entities']
                    if e['regions'] and e['regions'] != [-1])

region_dist = Counter()
for s in structured_output:
    for e in s['entities']:
        for r in e['regions']:
            if r >= 0:
                region_dist[r] += 1


coverage_pct = with_region / max(total_ent, 1) * 100
print(f"\n--- Mapping Coverage ---")
print(f"  Entities with region:    {with_region}/{total_ent} ({coverage_pct:.1f}%)")
print(f"  Entities without region: {total_ent - with_region}")
print(f"  (Unmapped = ltokens like 'unchanged','stable','no evidence' — EXPECTED)")
if coverage_pct < 10.0:
    print(f"  ⚠ WARNING: Coverage only {coverage_pct:.1f}% — suspiciously low.")
    print(f"  Check DEBUG label output above. Expected: ['ANAT-DP','OBS-DP',...]")
elif coverage_pct < 30.0:
    print(f"  ⚠ Coverage {coverage_pct:.1f}% — lower than ideal (target ≥30%).")
else:
    print(f"  ✅ Coverage {coverage_pct:.1f}% — acceptable for generated reports.")


unmapped_terms = Counter()
for s in structured_output:
    for e in s['entities']:
        if not e['regions'] or e['regions'] == [-1]:
            unmapped_terms[e['tokens'].lower()] += 1
if unmapped_terms:
    print(f"\n--- Top 15 Unmapped Terms (informational) ---")
    for term, cnt in unmapped_terms.most_common(15):
        print(f"  {term:30s}  {cnt}")


print(f"\n--- ImaGenome Integration (Cell 4b) ---")
if OBS_REGION_TABLE_EMPIRICAL:
    print(f"  ✅ Empirical obs→region table: {len(OBS_REGION_TABLE_EMPIRICAL)} evidence-grounded terms (Step 3a)")
    print(f"  ✅ Per-study GT index: {len(imagenome_study_regions)} test studies matched (4th pass)")
    print(f"  ✅ Entities resolved via 4th pass (ImaGenome GT): {imagenome_sourced_count}")

print(f"\n--- Entity Type Stats ---")
print(f"  Anatomy:           {anatomy_count}")
print(f"  Obs (present):     {obs_present}")
print(f"  Obs (absent):      {obs_absent}")
print(f"  Obs (uncertain):   {obs_uncertain}")
print(f"\n--- Region Distribution ---")
for rid in sorted(region_dist.keys()):
    rname = REGION_NAMES.get(rid, f"Region {rid}")
    print(f"  {rname:20s}  {region_dist[rid]:5d}")

print("=" * 60)
print("ENTITY EXTRACTION + REGION MAPPING — SUMMARY")
print("=" * 60)
print(f"  Total entities:    {total_ent} across {len(structured_output)} reports")
print(f"  Avg per report:    {total_ent / len(structured_output):.1f}")
print(f"\n  --- Entity Types ---")
print(f"  Anatomy:           {anatomy_count}")
print(f"  Obs (present):     {obs_present}")
print(f"  Obs (absent):      {obs_absent}")
print(f"  Obs (uncertain):   {obs_uncertain}")
print(f"\n  --- Region Coverage ---")
print(f"  With region:       {with_region}/{total_ent} "
      f"({with_region / max(total_ent,1) * 100:.1f}%)")
print(f"\n  --- Region Distribution ---")
for r_id in sorted(region_dist.keys()):
    print(f"  {r_id} {REGION_NAMES[r_id]:20s}: {region_dist[r_id]}")


print(f"\n--- Sample: {structured_output[0]['study_id']} ---")
for e in structured_output[0]['entities'][:8]:
    rnames = ', '.join(REGION_NAMES.get(r, '?') for r in e['regions'] if r >= 0)
    rnames = rnames or 'N/A'
    print(f"  [{e['entity_id']}] '{e['tokens']}' "
          f"— {e['type']}::{e['tag']}  → {rnames}")

In [ ]:
AUDIT_N = min(50, len(structured_output))  
audit_rng = random.Random(SEED)
audit_indices = sorted(audit_rng.sample(range(len(structured_output)), AUDIT_N))

audit_rows = []
for idx in audit_indices:
    s = structured_output[idx]
    for e in s['entities']:
        region_names = ', '.join(
            REGION_NAMES.get(r, f'?{r}') for r in e['regions'] if r >= 0
        ) or 'UNMAPPED'
        audit_rows.append({
            'report_idx': idx,
            'study_id': s['study_id'],
            'report_snippet': s['report'][:120],
            'entity_id': e['entity_id'],
            'tokens': e['tokens'],
            'type': e['type'],
            'tag': e['tag'],
            'regions': str(e['regions']),
            'region_names': region_names,
            'relations': str(e['relations']),
            
            'extraction_correct': '',
            'anatomy_correct': '',
            'laterality_correct': '',
            'uncertainty_correct': '',
            'region_mapping_correct': '',
            'notes': '',
        })

audit_df = pd.DataFrame(audit_rows)
audit_path = os.path.join(OUTPUT_DIR, 'audit_sample.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)
audit_df.to_csv(audit_path, index=False)

print(f"✅ Audit sample exported: {audit_path}")
print(f"   {AUDIT_N} reports, {len(audit_rows)} entity rows")
print(f"   Columns for manual annotation: extraction_correct, anatomy_correct,")
print(f"   laterality_correct, uncertainty_correct, region_mapping_correct, notes")
print(f"\n--- Sample audit rows ---")
print(audit_df[['study_id', 'tokens', 'type', 'tag', 'region_names']].head(10).to_string())

In [ ]:
failure_taxonomy = {
    'mapped_exact': 0,
    'mapped_bilateral': 0,
    'mapped_obs_table': 0,
    'mapped_heuristic': 0,
    'mapped_relation': 0,
    'unmapped': 0,
    
    'anatomy_missing': 0,
    'relation_missing': 0,
    'laterality_used': 0,
    'diffuse_global': 0,
    'uncertain_absent': 0,
}

failure_examples = {k: [] for k in failure_taxonomy}

for s in structured_output:
    
    report_anatomy_ids = set()
    for e in s['entities']:
        if e['type'] == 'ANAT':
            report_anatomy_ids.add(e['entity_id'])

    for e in s['entities']:
        text = e['tokens'].lower().strip()

        
        if 'absent' in e['tag'] or 'uncertain' in e['tag']:
            failure_taxonomy['uncertain_absent'] += 1

        
        if e['type'] == 'ANAT':
            
            if text in ANATOMY_REGION_TABLE:
                failure_taxonomy['mapped_exact'] += 1
            elif text in BILATERAL_LUNG_TERMS:
                failure_taxonomy['mapped_bilateral'] += 1
                failure_taxonomy['diffuse_global'] += 1
            elif e['regions'] != [-1]:
                failure_taxonomy['mapped_heuristic'] += 1
                if 'right' in text or 'left' in text:
                    failure_taxonomy['laterality_used'] += 1
            else:
                failure_taxonomy['unmapped'] += 1
                if len(failure_examples['unmapped']) < 10:
                    failure_examples['unmapped'].append(text)

        elif e['type'] == 'OBS':
            if e['regions'] and e['regions'] != [-1]:
                
                has_located_at = any(
                    r[0] == 'located_at' and str(r[1]) in report_anatomy_ids
                    for r in e['relations']
                )
                if has_located_at:
                    failure_taxonomy['mapped_relation'] += 1
                elif text in OBSERVATION_REGION_TABLE:
                    failure_taxonomy['mapped_obs_table'] += 1
                    if OBSERVATION_REGION_TABLE[text] == [0, 1]:
                        failure_taxonomy['diffuse_global'] += 1
                else:
                    failure_taxonomy['mapped_heuristic'] += 1
            else:
                failure_taxonomy['unmapped'] += 1
                failure_taxonomy['anatomy_missing'] += 1
                if len(failure_examples['anatomy_missing']) < 10:
                    failure_examples['anatomy_missing'].append(
                        f"{text} (study={s['study_id']})"
                    )
                
                if report_anatomy_ids:
                    failure_taxonomy['relation_missing'] += 1
                    if len(failure_examples['relation_missing']) < 10:
                        failure_examples['relation_missing'].append(
                            f"{text} (study={s['study_id']}, "
                            f"anatomy_in_report={len(report_anatomy_ids)})"
                        )


print("=" * 65)
print("MAPPING FAILURE TAXONOMY")
print("=" * 65)

print("\n  Mapping Resolution Methods:")
for key in ['mapped_exact', 'mapped_bilateral', 'mapped_obs_table',
            'mapped_heuristic', 'mapped_relation', 'unmapped']:
    pct = failure_taxonomy[key] / max(total_ent, 1) * 100
    marker = '✅' if key != 'unmapped' else '❌'
    print(f"  {marker} {key:22s}  {failure_taxonomy[key]:5d}  ({pct:5.1f}%)")

print(f"\n  Specific Failure/Edge Cases:")
for key in ['anatomy_missing', 'relation_missing', 'laterality_used',
            'diffuse_global', 'uncertain_absent']:
    print(f"     {key:22s}  {failure_taxonomy[key]:5d}")


for fail_key in ['unmapped', 'anatomy_missing', 'relation_missing']:
    if failure_examples[fail_key]:
        print(f"\n  Examples [{fail_key}]:")
        for ex in failure_examples[fail_key][:5]:
            print(f"    - {ex}")


failure_analysis = {
    'taxonomy': failure_taxonomy,
    'total_entities': total_ent,
    'coverage_pct': round(coverage_pct, 1),
    'examples': {k: v for k, v in failure_examples.items() if v},
}
failure_path = os.path.join(OUTPUT_DIR, 'failure_analysis.json')
with open(failure_path, 'w') as f:
    json.dump(failure_analysis, f, indent=2)
print(f"\n✅ Saved: {failure_path}")


print("\n" + "=" * 65)
print("MANUSCRIPT WORDING PATCH — Module 3")
print("=" * 65)
print(.format(
    n_anatomy=len(ANATOMY_REGION_TABLE),
    n_obs=len(OBSERVATION_REGION_TABLE),
    cov=f"{coverage_pct:.1f}",
    unmapped=total_ent - with_region,
    lat=failure_taxonomy['laterality_used'],
    diff=failure_taxonomy['diffuse_global'],
))

In [ ]:
print("=" * 65)
print("IMAGENOME GROUND-TRUTH VALIDATION (IAR)")
print("=" * 65)


imagenome_agreement        = None
imagenome_validation_rows  = []

if not imagenome_study_regions:
    print("  ⚠ ImaGenome per-study GT index is empty.")
    print(f"  ImaGenome scene graphs loaded: {imagenome_n_loaded:,}")
    print(f"  Unique obs terms from attributes: {len(imagenome_obs_region_freq)}")
    if len(imagenome_obs_region_freq) == 0:
        print("  DIAGNOSIS: Scene graph JSONs in this dataset version do not contain")
        print("  embedded attribute labels. The anatomy bounding-box definitions from")
        print("  IMAGENOME_OBJ_TO_REGION are still used in ANATOMY_REGION_TABLE.")
        print("  Entity-to-region mapping runs on heuristic tables (see Cell 5 coverage).")
    else:
        print("  DIAGNOSIS: study_id format mismatch — ImaGenome IDs do not match test set.")
        print(f"  Sample test IDs: {[str(s['study_id']) for s in structured_output[:5]]}")
        print(f"  Sample imagenome obs keys: {list(imagenome_obs_region_freq.keys())[:5]}")
    print("  Skipping IAR — data limitation, not a code bug.")
    print("  Module 3 outputs are complete: entities_with_regions.json + F1-RadGraph.")
    imagenome_agreement = None
if imagenome_study_regions:
    
    matched_studies = [s for s in structured_output
                       if str(s['study_id']) in imagenome_study_regions]
    unmatched = len(structured_output) - len(matched_studies)

    print(f"  Test reports with ImaGenome GT:   {len(matched_studies)} / {len(structured_output)}")
    print(f"  Test reports without ImaGenome:   {unmatched} "
          f"(studies not in silver dataset or disjoint split)")

    
    n_verifiable        = 0   
    n_agree             = 0   
    n_region_mismatch   = 0   
    n_not_in_imagenome  = 0   

    agree_by_region: Counter = Counter()   
    total_by_region: Counter = Counter()   

    for s in matched_studies:
        sid_str = str(s['study_id'])
        sg_data = imagenome_study_regions[sid_str]   

        
        _finding_to_sg_regions: dict = defaultdict(list)
        for rid, findings in sg_data.items():
            rid_int = int(rid) if isinstance(rid, str) else rid
            for f in findings:
                _finding_to_sg_regions[f].append(rid_int)

        for e in s['entities']:
            if e['type'] != 'OBS':
                continue
            pred_regions = [r for r in (e['regions'] or []) if r >= 0]
            if not pred_regions:
                continue

            finding = e['tokens'].lower().strip()

            
            sg_region_ids: list = []
            for sg_f, sg_rids in _finding_to_sg_regions.items():
                if finding == sg_f or finding in sg_f or sg_f in finding:
                    sg_region_ids.extend(sg_rids)
            sg_region_ids = list(set(sg_region_ids))

            if not sg_region_ids:
                n_not_in_imagenome += 1
                continue   

            n_verifiable += 1
            overlap = set(pred_regions) & set(sg_region_ids)
            if overlap:
                n_agree += 1
                for r in overlap:
                    agree_by_region[r] += 1
            else:
                n_region_mismatch += 1

            for r in pred_regions:
                total_by_region[r] += 1

            imagenome_validation_rows.append({
                'study_id':       s['study_id'],
                'finding':        e['tokens'],
                'tag':            e['tag'],
                'pred_regions':   pred_regions,
                'sg_regions':     sg_region_ids,
                'agreed':         bool(overlap),
                'imagenome_src':  e.get('imagenome_sourced', False),
            })

    iar = n_agree / max(n_verifiable, 1) * 100

    
    print(f"\n  {'─'*55}")
    print(f"  ImaGenome Agreement Rate (IAR)     = {iar:.1f}%")
    print(f"  {'─'*55}")
    print(f"  Verifiable entity-region pairs:    {n_verifiable:,}")
    print(f"  Agreements (pred ∩ GT ≠ ∅):        {n_agree:,}  ({iar:.1f}%)")
    print(f"  Region mismatches:                 {n_region_mismatch:,}  "
          f"({n_region_mismatch/max(n_verifiable,1)*100:.1f}%)")
    print(f"  Not annotated in ImaGenome GT:     {n_not_in_imagenome:,}")

    
    print(f"\n  --- IAR by Anatomical Region ---")
    print(f"  {'Region':<22}  {'Agree':>6}  {'Total':>6}  {'IAR':>7}  Progress")
    print(f"  {'─'*62}")
    for rid in sorted(total_by_region.keys()):
        rname = REGION_NAMES.get(rid, f'R{rid}')
        tot   = total_by_region[rid]
        agr   = agree_by_region.get(rid, 0)
        pct   = agr / max(tot, 1) * 100
        bar   = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f"  {rname:<22}  {agr:6d}  {tot:6d}  {pct:6.1f}%  {bar}")

    
    if imagenome_sourced_count > 0 or OBS_REGION_TABLE_EMPIRICAL:
        emp_agree   = sum(1 for r in imagenome_validation_rows
                          if r['agreed'] and not r['imagenome_src'])
        gt_agree    = sum(1 for r in imagenome_validation_rows
                          if r['agreed'] and r['imagenome_src'])
        print(f"\n  --- Agreed pairs by region-source ---")
        print(f"  Cascade table (empirical/hand-curated): {emp_agree:,}")
        print(f"  ImaGenome GT 4th-pass:                  {gt_agree:,}")

    
    mismatches = [r for r in imagenome_validation_rows if not r['agreed']]
    if mismatches:
        print(f"\n  --- Sample Region Mismatches (first 8) ---")
        print(f"  {'Finding':<28}  {'Predicted':<20}  {'ImaGenome GT'}")
        print(f"  {'─'*72}")
        for row in mismatches[:8]:
            pred_names = ', '.join(REGION_NAMES.get(r, f'R{r}')
                                   for r in row['pred_regions'])
            sg_names   = ', '.join(REGION_NAMES.get(r, f'R{r}')
                                   for r in row['sg_regions'])
            print(f"  {row['finding']:<28}  {pred_names:<20}  {sg_names}")

    
    print(f"\n  {'='*65}")
    print(f"  THESIS §3 METHODS — RECOMMENDED WORDING:")
    print(f"  {'='*65}")
    print(f)

    
    imagenome_agreement = {
        'n_matched_studies':      len(matched_studies),
        'n_unmatched_studies':    unmatched,
        'n_verifiable':           n_verifiable,
        'n_agree':                n_agree,
        'n_mismatch':             n_region_mismatch,
        'n_not_in_imagenome':     n_not_in_imagenome,
        'iar_pct':                round(iar, 2),
        'imagenome_sourced_count': imagenome_sourced_count,
        'empirical_table_terms':  len(OBS_REGION_TABLE_EMPIRICAL),
        'imagenome_sg_loaded':    imagenome_n_loaded,
        'agree_by_region': {
            REGION_NAMES.get(k, f'R{k}'): v for k, v in agree_by_region.items()
        },
        'total_by_region': {
            REGION_NAMES.get(k, f'R{k}'): v for k, v in total_by_region.items()
        },
    }
    print(f"\n✅ ImaGenome Agreement Rate (IAR) = {iar:.1f}%  — ready for thesis §3/§4.")

In [ ]:
from radgraph import F1RadGraph


gen_reports = [
    str(r).strip() if pd.notna(r) and str(r).strip() else "normal"
    for r in predictions_df['report'].tolist()
]
gt_reports_raw = [
    str(r).strip() if pd.notna(r) and str(r).strip() else ""
    for r in predictions_df['ground_truth'].tolist()
]

_has_gt = sum(1 for r in gt_reports_raw if r)
print(f"GT report availability: {_has_gt}/{len(gt_reports_raw)} reports have text")

assert _has_gt > 0, (
    "STOP: ground_truth column is empty — F1-RadGraph cannot run.\n"
    "Cell 2 must populate ground_truth from annotation.json.\n"
    f"Check that annotation.json study_ids match predictions_df study_ids."
)



valid_indices = [i for i, r in enumerate(gt_reports_raw) if r]
gen_valid = [gen_reports[i] for i in valid_indices]
gt_valid  = [gt_reports_raw[i] for i in valid_indices]
print(f"Computing F1-RadGraph on {len(valid_indices)} pairs with GT text...")
print(f"  (Excluded {len(gen_reports) - len(valid_indices)} reports without GT in annotation.json)")
start = time.time()
f1_device = 0 if NUM_GPUS >= 1 else -1
f1radgraph = F1RadGraph(reward_level="all", model_type="radgraph-xl", cuda=f1_device)
mean_reward, reward_list, hyp_annotations, ref_annotations = f1radgraph(
    hyps=gen_valid, refs=gt_valid
)
rg_e, rg_er, rg_bar_er = mean_reward
elapsed = time.time() - start
print(f"\n✅ F1-RadGraph computed in {elapsed:.1f}s")
rg_metrics = {
    'RG_E':      float(rg_e),
    'RG_ER':     float(rg_er),
    'RG_bar_ER': float(rg_bar_er),
}
rg_er_per_sample = reward_list[1]

print("\n" + "=" * 60)
print("F1-RADGRAPH METRICS — R2Gen Baseline vs Ground Truth")
print("=" * 60)
print(f"  RG_E     (Entity F1):            {rg_metrics['RG_E']:.4f}")
print(f"  RG_ER    (Entity+Relation F1):   {rg_metrics['RG_ER']:.4f}")
print(f"  RG_bar_ER (Macro-avg E+R F1):    {rg_metrics['RG_bar_ER']:.4f}")
print("  Note: RG_ER is the standard metric in RRG papers.")
print(f"\n  --- Per-Sample RG_ER Distribution ---")
print(f"  Mean:   {np.mean(rg_er_per_sample):.4f}")
print(f"  Median: {np.median(rg_er_per_sample):.4f}")
print(f"  Std:    {np.std(rg_er_per_sample):.4f}")
print(f"  Min:    {np.min(rg_er_per_sample):.4f}")
print(f"  Max:    {np.max(rg_er_per_sample):.4f}")
rg_metrics_path = os.path.join(OUTPUT_DIR, 'radgraph_metrics.json')
with open(rg_metrics_path, 'w') as f:
    json.dump(rg_metrics, f, indent=2)
print(f"\n✅ Saved: {rg_metrics_path}")

In [ ]:
print("=" * 60)
print("QUALITY CHECKS")
print("=" * 60)


assert len(structured_output) == len(predictions_df), \
    f"Mismatch: {len(structured_output)} vs {len(predictions_df)}"
print(f"✅ All {len(structured_output)} reports have entity annotations")


avg_ent = total_ent / len(structured_output)
assert avg_ent > 1.0, f"Suspiciously low avg entities: {avg_ent:.1f}"
print(f"✅ Avg entities/report = {avg_ent:.1f} (reasonable for radiology)")



assert obs_present > 0, "No present observations found"
print(f"✅ Observation distribution: {obs_present} present, "
      f"{obs_absent} absent, {obs_uncertain} uncertain")


coverage = with_region / max(total_ent, 1) * 100
print(f"✅ Region mapping coverage: {coverage:.1f}%")


empty_reports = sum(1 for s in structured_output
                    if s['num_entities'] == 0 and len(s['report'].split()) > 3)
print(f"{'✅' if empty_reports == 0 else '⚠️'} "
      f"Reports with 0 entities (excluding very short): {empty_reports}")


assert rg_metrics.get('RG_ER') is not None, (
    "STOP: F1-RadGraph RG_ER is None — Cell 6 did not complete.\n"
    "Ensure Cell 2 (annotation.json) ran successfully first."
)
assert rg_metrics['RG_ER'] > 0.0, f"STOP: F1-RadGraph RG_ER = 0 — something is wrong"
print(f"✅ Check 6 PASSED: F1-RadGraph RG_ER = {rg_metrics['RG_ER']:.4f}")


obs_counter = Counter()
for s in structured_output:
    for e in s['entities']:
        if e['type'] == 'OBS':
            obs_counter[e['tokens'].lower()] += 1

print(f"\n--- Top 15 Observations ---")
for tok, cnt in obs_counter.most_common(15):
    print(f"  {tok:30s}  {cnt}")


anat_counter = Counter()
for s in structured_output:
    for e in s['entities']:
        if e['type'] == 'ANAT':
            anat_counter[e['tokens'].lower()] += 1

print(f"\n--- Top 10 Anatomy ---")
for tok, cnt in anat_counter.most_common(10):
    region_ids = map_anatomy_to_region(tok)
    rnames = ', '.join(REGION_NAMES.get(r, '?') for r in region_ids if r >= 0) or 'N/A'
    print(f"  {tok:25s}  {cnt:4d}  → {rnames}")


by_count = sorted(structured_output, key=lambda x: x['num_entities'], reverse=True)
print(f"\n--- Reports with Most Entities ---")
for s in by_count[:5]:
    print(f"  {str(s['study_id']):20s}  {s['num_entities']} entities")
    print(f"    {s['report'][:90]}...")

In [ ]:
import shutil


assert 'm2_metrics' in dir() and m2_metrics, (
    "STOP: m2_metrics not defined. Cell 2 must complete successfully first."
)


entities_path = os.path.join(OUTPUT_DIR, 'entities_with_regions.json')
with open(entities_path, 'w') as f:
    json.dump(structured_output, f, indent=2)
print(f"✅ Saved: {entities_path}  ({len(structured_output)} reports)")


raw_path = os.path.join(OUTPUT_DIR, 'radgraph_entities.json')
with open(raw_path, 'w') as f:
    json.dump(all_annotations, f, indent=2)
print(f"✅ Saved: {raw_path}")


print(f"✅ Saved: {rg_metrics_path}")


analysis = {
    'total_reports': len(structured_output),
    'total_entities': total_ent,
    'avg_entities_per_report': round(total_ent / len(structured_output), 2),
    'entity_type_counts': {
        'anatomy': anatomy_count,
        'observation_present': obs_present,
        'observation_absent': obs_absent,
        'observation_uncertain': obs_uncertain,
    },
    'region_mapping': {
        'entities_with_region': with_region,
        'coverage_pct': round(with_region / max(total_ent, 1) * 100, 1),
        'distribution': {REGION_NAMES[k]: v for k, v in sorted(region_dist.items())},
    },
    'f1_radgraph': rg_metrics,
    'module2_path_mention_rate': m2_metrics['mean_pathology_mention_rate'],
}
analysis_path = os.path.join(OUTPUT_DIR, 'module3_analysis.json')
with open(analysis_path, 'w') as f:
    json.dump(analysis, f, indent=2)
print(f"✅ Saved: {analysis_path}")


zip_path = '/kaggle/working/module3_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✅ Saved: {zip_path}.zip")


print("\n" + "=" * 60)
print("MODULE 3 COMPLETE — RadGraph Entity Extraction (MIMIC-CXR)")
print("=" * 60)
print(f)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


IEEE_SINGLE_COL = 3.5   
IEEE_DOUBLE_COL = 7.16  

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.5,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.6,
    'lines.linewidth': 1.2,
    'patch.linewidth': 0.5,
})

C_BLUE = '
C_RED = '
C_GREEN = '
C_PURPLE = '
C_ORANGE = '
C_GRAY = '
C_TEAL = '

FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)




fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 2.8))

type_counts = {
    'Anatomy': anatomy_count,
    'Obs\n(Present)': obs_present,
    'Obs\n(Absent)': obs_absent,
    'Obs\n(Uncertain)': obs_uncertain,
}
colors_types = [C_BLUE, C_GREEN, C_RED, C_ORANGE]

bars = ax.bar(type_counts.keys(), type_counts.values(), color=colors_types,
              edgecolor='white', linewidth=0.5, width=0.6)
for bar, val in zip(bars, type_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{val:,}\n({val/total_ent*100:.1f}%)', ha='center', va='bottom', fontsize=6)

ax.set_ylabel('Count')
ax.set_title(f'Entity Type Distribution (N={total_ent:,}, {len(structured_output)} reports)')
ax.set_ylim([0, max(type_counts.values()) * 1.25])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_entity_types.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_entity_types.pdf'))
plt.show()
print("✅ Saved: fig_m3_entity_types.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.0))

rids = sorted(region_dist.keys())
rnames = [REGION_NAMES.get(r, f'Region {r}') for r in rids]
rcounts = [region_dist[r] for r in rids]
colors_reg = [C_BLUE, C_TEAL, C_RED, C_PURPLE, C_ORANGE, C_GREEN, '

bars = ax.bar(range(len(rids)), rcounts,
              color=[colors_reg[i % len(colors_reg)] for i in range(len(rids))],
              edgecolor='white', linewidth=0.5, width=0.65)

for bar, val in zip(bars, rcounts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}\n({val/sum(rcounts)*100:.1f}%)', ha='center', va='bottom', fontsize=6)

ax.set_xticks(range(len(rids)))
ax.set_xticklabels(rnames, rotation=30, ha='right')
ax.set_ylabel('Entity Count')
ax.set_title('Entity Distribution Across 8 Anatomical Regions')
ax.set_ylim([0, max(rcounts) * 1.25])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_region_distribution.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_region_distribution.pdf'))
plt.show()
print("✅ Saved: fig_m3_region_distribution.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 2.8))

ent_per_report = [s['num_entities'] for s in structured_output]
avg_ent_pr = np.mean(ent_per_report)
med_ent_pr = np.median(ent_per_report)

ax.hist(ent_per_report, bins=np.arange(0, max(ent_per_report)+2, 2),
        color=C_BLUE, alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(avg_ent_pr, color=C_RED, linestyle='--', linewidth=1.2,
           label=f'Mean = {avg_ent_pr:.1f}')
ax.axvline(med_ent_pr, color=C_GREEN, linestyle=':', linewidth=1.2,
           label=f'Median = {med_ent_pr:.0f}')

ax.set_xlabel('Number of Entities per Report')
ax.set_ylabel('Number of Reports')
ax.set_title('Entity Count Distribution per Report')
ax.legend(framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_entities_per_report.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_entities_per_report.pdf'))
plt.show()
print("✅ Saved: fig_m3_entities_per_report.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 3.5))

obs_ctr = Counter()
for s in structured_output:
    for e in s['entities']:
        if e['type'] == 'OBS':
            obs_ctr[e['tokens'].lower()] += 1

top_obs = obs_ctr.most_common(15)
obs_names = [o[0] for o in top_obs][::-1]
obs_vals = [o[1] for o in top_obs][::-1]

bars = ax.barh(range(len(obs_names)), obs_vals, color=C_BLUE, edgecolor='white', linewidth=0.3, height=0.65)
for bar, val in zip(bars, obs_vals):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val}', ha='left', va='center', fontsize=6)

ax.set_yticks(range(len(obs_names)))
ax.set_yticklabels(obs_names)
ax.set_xlabel('Frequency')
ax.set_title('Top-15 Observation Entities Extracted by RadGraph-XL')

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_top_observations.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m3_top_observations.pdf'))
plt.show()
print("✅ Saved: fig_m3_top_observations.{png,pdf}")





if rg_metrics.get('RG_ER') is None:
    print("⚠️ Fig 5 skipped — F1-RadGraph not computed (GT not in MIMIC-CXR-JPG)")
else:
    rg_er_arr = np.array(rg_er_per_sample)
    fig_rg, (ax1, ax2) = plt.subplots(1, 2, figsize=(IEEE_DOUBLE_COL, 2.8),
                                       gridspec_kw={'width_ratios': [2, 1]})
    ax1.hist(rg_er_arr, bins=30, color=C_BLUE, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax1.axvline(np.mean(rg_er_arr), color=C_RED, linestyle='--', linewidth=1.2,
                label=f'Mean = {np.mean(rg_er_arr):.4f}')
    ax1.axvline(np.median(rg_er_arr), color=C_GREEN, linestyle=':', linewidth=1.2,
                label=f'Median = {np.median(rg_er_arr):.4f}')
    ax1.set_xlabel('F1-RadGraph (Entity + Relation)')
    ax1.set_ylabel('Number of Reports')
    ax1.set_title('(a) Per-Sample F1-RadGraph Distribution')
    ax1.legend(framealpha=0.9)
    bp = ax2.boxplot(rg_er_arr, vert=True, patch_artist=True,
                     boxprops=dict(facecolor=C_BLUE, alpha=0.5),
                     medianprops=dict(color=C_RED, linewidth=1.5),
                     whiskerprops=dict(color=C_GRAY),
                     flierprops=dict(marker='o', markerfacecolor=C_GRAY, markersize=2, alpha=0.5))
    ax2.set_xticklabels(['RG_ER'])
    ax2.set_ylabel('F1-RadGraph (Entity + Relation)')
    ax2.set_title('(b) Box Plot')
    q1, q2, q3 = np.percentile(rg_er_arr, [25, 50, 75])
    ax2.annotate(f'Q1={q1:.3f}\nQ2={q2:.3f}\nQ3={q3:.3f}',
                 xy=(1.15, q2), fontsize=6, va='center',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor=C_GRAY))
    plt.tight_layout(w_pad=2)
    fig_rg.savefig(os.path.join(FIG_DIR, 'fig_m3_radgraph_distribution.png'))
    fig_rg.savefig(os.path.join(FIG_DIR, 'fig_m3_radgraph_distribution.pdf'))
    plt.show()
    print("✅ Saved: fig_m3_radgraph_distribution.{png,pdf}")




try:
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.5))

    top10_obs = [o[0] for o in obs_ctr.most_common(10)]
    matrix = np.zeros((len(top10_obs), len(REGION_NAMES)))

    for s in structured_output:
        for e in s['entities']:
            if e['type'] == 'OBS' and e['tokens'].lower() in top10_obs:
                row = top10_obs.index(e['tokens'].lower())
                for r in e['regions']:
                    if 0 <= r < len(REGION_NAMES):
                        matrix[row, r] += 1

    region_labels = [REGION_NAMES[i] for i in range(len(REGION_NAMES))]
    sns.heatmap(matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
                xticklabels=region_labels, yticklabels=top10_obs,
                cbar_kws={'label': 'Count'}, linewidths=0.5, linecolor='white')

    ax.set_xlabel('Anatomical Region')
    ax.set_ylabel('Observation Entity')
    ax.set_title('Observation–Region Co-occurrence Matrix (Top-10 Observations)')
    plt.xticks(rotation=35, ha='right')

    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'fig_m3_obs_region_heatmap.png'))
    fig.savefig(os.path.join(FIG_DIR, 'fig_m3_obs_region_heatmap.pdf'))
    plt.show()
    print("✅ Saved: fig_m3_obs_region_heatmap.{png,pdf}")
except ImportError:
    print("⚠️ seaborn not installed — skipping heatmap (pip install seaborn)")




print("\n" + "=" * 60)
print("MODULE 3 — PUBLICATION FIGURES SUMMARY")
print("=" * 60)
print(f)